In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.embeddings import OllamaEmbeddings
from langchain_community.vectorstores import FAISS

In [3]:
loader = TextLoader("../data/speech.txt")
documents = loader.load()
text_splitter = CharacterTextSplitter( chunk_size=500, chunk_overlap=20)
splits = text_splitter.split_documents(documents)

In [5]:
len(splits)

12

In [6]:
embedder = OllamaEmbeddings(model="gemma:2b")

/var/folders/fy/lwhjv2_15kvf5gcd_746srd80000gn/T/ipykernel_14986/1513822933.py:1: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaEmbeddings``.
  embedder = OllamaEmbeddings(model="gemma:2b")


In [7]:
db = FAISS.from_documents(splits, embedder)

In [8]:
# querying the db
query = "What is the speech about?"
docs = db.similarity_search(query)
docs[0].page_content

'This document, while simple, serves as a practical test input for validating the behavior of document loaders in a RAG pipeline. It ensures that paragraph-level semantics are preserved and that downstream components receive meaningful input. By using such realistic text samples, developers can identify weaknesses in their ingestion and retrieval strategies and refine their systems accordingly.'

In [11]:
retriever = db.as_retriever()
docs = retriever.invoke(query)
docs[0].page_content

'This document, while simple, serves as a practical test input for validating the behavior of document loaders in a RAG pipeline. It ensures that paragraph-level semantics are preserved and that downstream components receive meaningful input. By using such realistic text samples, developers can identify weaknesses in their ingestion and retrieval strategies and refine their systems accordingly.'

In [17]:
# querying the db
query = "What is the speech about?"
docs = db.similarity_search_with_score(query)
len(docs[0])

2

In [19]:
docs = db.similarity_search_by_vector(embedder.embed_query(query))
len(docs)

4

In [20]:
db.save_local("faiss_index")

In [22]:
new_df = FAISS.load_local("faiss_index", embedder, allow_dangerous_deserialization=True)